# fuselk Vision Overview

**Fusion Unified Simulation, ELM Learning & Kinetics** — the open-source autopilot for next-generation tokamaks.

This notebook maps the [VISION.md](../VISION.md) architecture to runnable code. Companion notebooks cover each pillar in depth:

| Notebook | Topic |
|----------|-------|
| `01_oil_water_barrier_math` | Coupled plasma/vapor PDEs, Peclet extraction |
| `02_helix_hqrm_diagnostics` | Phase-locked denoising, HQRM 7×7 lock |
| `03_muon_tritium_fuel_cycle` | Catalytic rate network, breeding blanket |
| `04_venturi_control_experiments` | Hierarchical RL, disruption fusion |
| `05_reactor_cell_digital_twin` | Closed-loop ReactorCell + FusionKPIs |
| **`06_unified_cross_domain_mathematics`** | **Full VISION equation sets, proofs, domain coupling** |

## The four existential bottlenecks (VISION.md)

1. **ELMs & disruptions** — MHD precursors terminate pulses
2. **Divertor melt** — localized exhaust (>10 MW/m²)
3. **Alpha sticking** — μ⁻ lost to (αμ)⁺ before catalytic breakeven
4. **Tritium self-sufficiency** — closed fuel cycle requires TBR > 1 and Pe > 1 extraction

> **Start here for theory:** `06_unified_cross_domain_mathematics.ipynb` lays out every governing equation, proof sketch, and cross-domain coupling from [VISION.md](../VISION.md). Notebooks 01–05 pair that theory with runnable experiments.


## End-to-end data flow

```
Diagnostics → HELIX/HQRM → Focal Heat Map → ELM Predictor → Venturi Controller → Actuators
                    ↓
              Physics PDE ← Breeding Blanket ← Muon Cycle
```

Each module is a Python package under `src/deepiri_fuselk/`. Experiments are registered in `experiments/registry.yaml`.

See [docs/DATA_PIPELINE.md](../docs/DATA_PIPELINE.md) for the full ingest story: MIT Open Density Limit CSV → normalized C-Mod HDF5 shots, plus synthetic `elm_corpus.npz` for training.


In [ ]:
import sys
from pathlib import Path

from deepiri_fuselk.data.notebook_loaders import resolve_repo_root

repo = resolve_repo_root()

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]

import deepiri_fuselk as fuselk
print(f'fuselk v{fuselk.__version__}')

## Data lake — real shots from `fuselk data fetch`

In [ ]:
from deepiri_fuselk.data.imas_loader import load_imas_hdf5
from deepiri_fuselk.data.notebook_loaders import (
    ensure_fetched_data,
    imas_to_synthetic_shot,
    list_shots,
    load_corpus_arrays,
    load_odl_meta,
    manifest_summary,
    resolve_data_root,
)

data_root = ensure_fetched_data(resolve_data_root(repo), n_shots=100, max_odl=50)
cmod_shots = list_shots(data_root, source="cmod")
syn_shots = list_shots(data_root, source="synthetic")
corpus = load_corpus_arrays(data_root)
manifest = manifest_summary(data_root)

print(f"Data lake: {data_root}")
print(f"  C-Mod (ODL) shots: {len(cmod_shots)}")
print(f"  Synthetic shots:   {len(syn_shots)}")
print(f"  ELM corpus frames: {len(corpus['labels'])} (elm_rate={corpus['elm_rate']:.2f})")
for row in manifest:
    print(f"  [{row['source']}] {row['status']} — {row['shots']} shots @ {row['fetched_at'][:19]}")


In [ ]:
# Peek at one real Alcator C-Mod discharge (MIT Open Density Limit DB)
cmod_path = cmod_shots[0]
cmod = load_imas_hdf5(cmod_path)
odl = load_odl_meta(cmod_path)

print(f"Shot {cmod.shot_id} on {cmod.device}")
print(f"  ne core: {cmod.ne_profile.values[0]:.2e} m⁻³")
print(f"  q edge:  {cmod.q_profile.values[-1]:.2f}")
if odl:
    print(f"  ODL density-limit phase rate: {odl['elm_event_rate']:.2%}")
    print(f"  Time points: {len(odl['density_limit_phase'])}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(cmod.ne_profile.radius, cmod.ne_profile.values, label="n_e")
axes[0].plot(cmod.Te_profile.radius, np.array(cmod.Te_profile.values) / 1e3, label="T_e [keV]")
axes[0].legend(); axes[0].set_xlabel("ρ"); axes[0].set_title("Fetched profiles")

axes[1].imshow(cmod.heat_field, origin="lower", cmap="inferno")
axes[1].set_title("ECE heat field (HELIX input)")

if odl and odl.get("time") is not None:
    axes[2].plot(odl["time"], odl["density"], label="line-averaged n")
    ax2 = axes[2].twinx()
    ax2.plot(odl["time"], odl["density_limit_phase"], "r.", alpha=0.5, label="DL phase")
    axes[2].set_xlabel("time [s]"); axes[2].legend(loc="upper left")
else:
    axes[2].text(0.5, 0.5, "no ODL trace", ha="center", va="center", transform=axes[2].transAxes)
axes[2].set_title("ODL precursor labels")

plt.tight_layout()
plt.show()


In [ ]:
registry_path = Path("experiments/registry.yaml")
if not registry_path.exists():
    registry_path = Path("../experiments/registry.yaml")

for line in registry_path.read_text().splitlines():
    if line.strip().startswith("- id:"):
        exp_id = line.split(":", 1)[1].strip()
        print(f"  • {exp_id}")


## Quick smoke test — all pillars on real + synthetic data

In [ ]:
from deepiri_fuselk.helix import HelixEngine
from deepiri_fuselk.muon import RateNetworkParams, run_rate_network
from deepiri_fuselk.physics import PDEParameters, peclet_criterion, solve_oil_water_steady
from deepiri_fuselk.sim import ReactorCell

params = PDEParameters()
steady = solve_oil_water_steady(n_grid=32, params=params)

# Real C-Mod heat field through HELIX (not synthetic generator)
ece = imas_to_synthetic_shot(cmod)
helix = HelixEngine().process(ece.heat_field, ece.raw_signal, ece.angles)

muon = run_rate_network(params=RateNetworkParams(R_photon=0.3), t_span=(0.0, 1e-5))
cell = ReactorCell(grid_size=16, train_elm=False)
cell.imas = cmod  # drive q/Te profiles from fetched shot
run = cell.run(n_steps=5, seed=1)

print(f"PDE converged: {steady.converged}, Peclet Pe = {peclet_criterion(params):.2f}")
print(f"HELIX SNR (C-Mod): {helix.phase_locked_snr:.2f}, ELM prob: {helix.elm_probability:.2f}")
print(f"Muon fusions/μ: {muon.fusions_per_muon:.1f}, breakeven: {muon.breakeven}")
print(f"Reactor fusion score (real profiles): {run.final_score:.3f}")
